In [1]:
import uuid
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql import Row


StatementMeta(, 26ce2b6b-e15e-44bd-8c46-315f2d6c48e1, 3, Finished, Available, Finished, False)

In [2]:
# Generate a unique run identifier using the given prefix and a UUID
def new_dq_run_id(prefix="dq"):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    unique = uuid.uuid4().hex[:8]
    return f"{prefix}_{timestamp}_{unique}"

StatementMeta(, 26ce2b6b-e15e-44bd-8c46-315f2d6c48e1, 4, Finished, Available, Finished, False)

In [3]:
# Build a standardized data quality result row with rule metadata, metric values, and evaluation status

def make_rule_result_row(
    dq_run_id,
    pipeline_run_id,
    run_ts,
    layer,
    table_name,
    rule_id,
    rule_name,
    rule_type,
    column_name,
    severity,
    dimension,
    total_count,
    failed_count,
    failed_rate,
    threshold_rate,
    status,
    rule_message
):
    return {
        "dq_run_id": dq_run_id,
        "pipeline_run_id": pipeline_run_id,
        "run_ts": run_ts,
        "layer": layer,
        "table_name": table_name,
        "rule_id": rule_id,
        "rule_name": rule_name,
        "rule_type": rule_type,
        "column_name": column_name,
        "severity": severity,
        "dimension": dimension,
        "total_count": int(total_count) if total_count is not None else None,
        "failed_count": int(failed_count) if failed_count is not None else None,
        "failed_rate": float(failed_rate) if failed_rate is not None else None,
        "threshold_rate": float(threshold_rate) if threshold_rate is not None else None,
        "status": status,
        "rule_message": rule_message
    }

StatementMeta(, 26ce2b6b-e15e-44bd-8c46-315f2d6c48e1, 5, Finished, Available, Finished, False)

In [4]:
def eval_standard_rule(
    df,
    rule,
    dq_run_id,
    pipeline_run_id,
    layer,
    table_name,
    run_ts
):
    total_count = df.count()

    failed_count = df.filter(rule["fail_cond"]).count()
    failed_rate = (failed_count)/ total_count if total_count > 0 else 0.0
    threshold_rate = rule["threshold_rate"]

    status = "FAIL" if failed_rate > threshold_rate else "PASS"

    rule_message = (
        f"{rule['rule_name']} failed_rate = {failed_rate:.6f}, "
        f"threshold_rate = {threshold_rate:.6f}"
    )

    return make_rule_result_row(
        dq_run_id = dq_run_id,
        pipeline_run_id = pipeline_run_id,
        run_ts = run_ts,
        layer = layer,
        table_name = table_name,
        rule_id = rule["rule_id"],
        rule_name = rule["rule_name"],
        rule_type = rule["rule_type"],
        column_name = rule.get("column_name"),
        severity = rule["severity"],
        dimension = rule["dimension"],
        total_count = total_count,
        failed_count = failed_count,
        failed_rate = failed_rate,
        threshold_rate = threshold_rate,
        status = status,
        rule_message = rule_message
    )
    



StatementMeta(, 26ce2b6b-e15e-44bd-8c46-315f2d6c48e1, 6, Finished, Available, Finished, False)

In [5]:
def eval_duplicate_key_rule(
    df,
    rule,
    dq_run_id,
    pipeline_run_id,
    layer,
    table_name,
    run_ts
):
    key_cols = rule["key_cols"]

    total_count = df.count()

    dup_df = (
        df.groupBy(*key_cols)
        .count()
        .filter(F.col("count") > 1)
    )

    duplicate_rows = dup_df.agg(F.sum("count").alias("dup_row_cnt")).collect()[0]["dup_row_cnt"]
    failed_count = duplicate_rows if duplicate_rows is not None else 0

    failed_rate = (failed_count / total_count) if total_count > 0 else 0.0
    threshold_rate = rule["threshold_rate"]

    status = "FAIL" if failed_rate > threshold_rate else "PASS"

    rule_message = (
        f"{rule['rule_name']} failed_rate={failed_rate:.6f}, "
        f"threshold_rate={threshold_rate:.6f}"
    )

    return make_rule_result_row(
        dq_run_id = dq_run_id,
        pipeline_run_id = pipeline_run_id,
        run_ts = run_ts,
        layer = layer,
        table_name = table_name,
        rule_id = rule["rule_id"],
        rule_name = rule["rule_name"],
        rule_type = rule["rule_type"],
        column_name = ",".join(key_cols),
        severity = rule["severity"],
        dimension = rule["dimension"],
        total_count = total_count,
        failed_count = failed_count,
        failed_rate = failed_rate,
        threshold_rate = threshold_rate,
        status = status,
        rule_message = rule_message
    )


StatementMeta(, 26ce2b6b-e15e-44bd-8c46-315f2d6c48e1, 7, Finished, Available, Finished, False)

In [6]:
def build_gate_result(
    rule_results,
    dq_run_id,
    pipeline_run_id,
    layer,
    table_name,
    run_ts
):
    total_rules = len(rule_results)
    passed_rules = sum(1 for r in rule_results if r["status"] == "PASS")
    failed_rules = sum(1 for r in rule_results if r["status"] == "FAIL")

    critical_failed_rules = sum(
        1 for r in rule_results
        if r["status"] == "FAIL" and r["severity"] == "CRITICAL"
    )

    major_failed_rules = sum(
        1 for r in rule_results
        if r["status"] == "FAIL" and r["severity"] == "MAJOR"
    )

    if critical_failed_rules > 0:
        decision = "BLOCKED"
        decision_reason = "At least one CRITICAL rule failed"
    elif major_failed_rules > 0:
        decision = "DEGRADED"
        decision_reason = "One or more Major rules failed"
    else:
        decision = "PASS"
        decision_reason = "All rules passed or only non-blocking issues found"
    
    return {
        "dq_run_id": dq_run_id,
        "pipeline_run_id": pipeline_run_id,
        "run_ts": run_ts,
        "layer": layer,
        "table_name": table_name,
        "total_rules": total_rules,
        "passed_rules": passed_rules,
        "failed_rules": failed_rules,
        "critical_failed_rules": critical_failed_rules,
        "major_failed_rules": major_failed_rules,
        "decision": decision,
        "decision_reason": decision_reason
    }

StatementMeta(, 26ce2b6b-e15e-44bd-8c46-315f2d6c48e1, 8, Finished, Available, Finished, False)